# sermon-insights — GPU transcription

YouTube blocks Colab IP ranges, so this notebook does **not** call `yt-dlp`.
Fetch audio on your own machine (throttled), transcode it, put it in a Drive folder, and point the two paths below at that folder.

Local prep, from the repo:

```bash
yt-dlp -f bestaudio --sleep-requests 2 --no-overwrites -o 'audio/%(id)s.%(ext)s' 'https://www.youtube.com/watch?v=VIDEO_ID'
ffmpeg -y -i audio/VIDEO_ID.webm -ac 1 -ar 16000 -c:a libopus -b:a 32k handoff/VIDEO_ID.opus
```

Speech files go in `sermons/`. Worship / singing goes in `worship/`.
There are no Drive file ids in this notebook. Share or upload the folder yourself.

Runtime: **T4 GPU**. Model: **faster-whisper `medium`**, `float16`, language `sw`.
Worship turns VAD off (`vad_filter=False`) and both modes set `condition_on_previous_text=False`.


In [ ]:
import subprocess
probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit('No GPU. Runtime → Change runtime type → T4, then run again.')
print(probe.stdout.splitlines()[0] if probe.stdout else 'GPU ok')


In [ ]:
!pip -q install -U faster-whisper 'nvidia-cublas-cu12==12.9.2.10' 'nvidia-cudnn-cu12==9.27.0.42'


In [ ]:
import ctypes, glob, os
import nvidia.cublas.lib, nvidia.cudnn.lib
libs = [os.path.dirname(nvidia.cublas.lib.__file__), os.path.dirname(nvidia.cudnn.lib.__file__)]
os.environ['LD_LIBRARY_PATH'] = ':'.join(libs + [os.environ.get('LD_LIBRARY_PATH', '')])
for directory in libs:
    for filename in sorted(glob.glob(directory + '/*.so*')):
        try:
            ctypes.CDLL(filename)
        except OSError:
            pass
print('CUDA libs preloaded from', libs)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change these to your own folders. Do not put file ids here.
AUDIO_DIR = '/content/drive/MyDrive/sermon-insights/audio'
OUT_DIR = '/content/drive/MyDrive/sermon-insights/transcripts'


In [ ]:
import json
from pathlib import Path
from faster_whisper import WhisperModel

audio_root = Path(AUDIO_DIR)
out_root = Path(OUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)
model = WhisperModel('medium', device='cuda', compute_type='float16')

def options(kind):
    base = dict(language='sw', beam_size=1, condition_on_previous_text=False)
    if kind == 'worship':
        base['vad_filter'] = False
        return base
    base['vad_filter'] = True
    base['vad_parameters'] = dict(threshold=0.3, min_silence_duration_ms=1000, speech_pad_ms=400)
    return base

files = sorted(audio_root.rglob('*.opus')) + sorted(audio_root.rglob('*.m4a')) + sorted(audio_root.rglob('*.webm'))
print(len(files), 'audio files')
for path in files:
    kind = 'worship' if 'worship' in path.parts else 'sermon'
    dest = out_root / kind / f'{path.stem}.json'
    if dest.is_file() and dest.stat().st_size > 0:
        print('skip', path.name)
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    segments, info = model.transcribe(str(path), **options(kind))
    payload = {
        'id': path.stem,
        'language': info.language,
        'model': 'medium',
        'source': f'whisper-{info.language}',
        'duration': info.duration,
        'segments': [
            {'start': float(row.start), 'end': float(row.end), 'text': row.text.strip()}
            for row in segments if row.text and row.text.strip()
        ],
    }
    dest.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
    print('wrote', dest, len(payload['segments']), 'lines', kind)
